In [1]:
!pip install -Uqqq pip
!pip install -qqq bitsandbytes==0.39.0
!pip install -qqq torch==2.0.1
!pip install -qqq -U git+https://github.com/huggingface/transformers.git@e03a9cc
!pip install -qqq -U git+https://github.com/huggingface/peft.git@42a184f
!pip install -qqq -U git+https://github.com/huggingface/accelerate.git@c9fbb71
!pip install -qqq datasets==2.12.0
!pip install -qqq loralib==0.1.1
!pip install -qqq einops==0.6.1
!pip install transformers==4.30
!pip uninstall bitsandbytes
!pip install bitsandbytes
!pip install -U datasets
!pip install rouge

In [4]:
# import os
# os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
# os.environ["CUDA_VISIBLE_DEVICES"]="0" #model will be trained on GPU 0

In [2]:
!apt-get update
!apt-get install cuda-toolkit-11-8
import os
os.environ["LD_LIBRARY_PATH"] += ":" + "/usr/local/cuda-11/lib64"
os.environ["LD_LIBRARY_PATH"] += ":" + "/usr/local/cuda-11.8/lib64"

/bin/bash: apt-get: command not found
/bin/bash: apt-get: command not found


In [2]:
# !apt-get update
# !apt-get install cuda-toolkit-11-8

In [2]:
import os

os.environ["LD_LIBRARY_PATH"] += ":" + "/usr/local/cuda-11/lib64"
os.environ["LD_LIBRARY_PATH"] += ":" + "/usr/local/cuda-11.8/lib64"

In [3]:
import os
os.environ["LD_LIBRARY_PATH"]

'/shared/centos7/cuda/11.2/lib64:/shared/centos7/anaconda3/2022.05/lib:/shared/centos7/nodejs/14.15.4/lib:/usr/local/cuda-11/lib64:/usr/local/cuda-11.8/lib64'

In [29]:
import json
import os
from pprint import pprint
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import load_dataset
from huggingface_hub import notebook_login
from peft import (
    LoraConfig,
    PeftConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training
)
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

## Loading the Model

In [2]:
MODEL_NAME = "vilsonrodrigues/falcon-7b-instruct-sharded"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    trust_remote_code=True,
    quantization_config=bnb_config
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

/home/justin.aj/.local/lib/python3.9/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/15 [00:00<?, ?it/s]

Some weights of FalconForCausalLM were not initialized from the model checkpoint at vilsonrodrigues/falcon-7b-instruct-sharded and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
def print_trainable_parameters(model):
  """
  Prints the number of trainable parameters in the model.
  """
  trainable_params = 0
  all_param = 0
  for _, param in model.named_parameters():
    all_param += param.numel()
    if param.requires_grad:
      trainable_params += param.numel()
  print(
      f"trainable params: {trainable_params} || all params: {all_param} || trainables%: {100 * trainable_params / all_param}"
  )

In [4]:
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [5]:
config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)
print_trainable_parameters(model)

trainable params: 4718592 || all params: 3613463424 || trainables%: 0.13058363808693696


# Test original model

In [6]:
prompt = """
<human>: Hey, I need help with something."
<assistant>:
""".strip()

In [7]:
generation_config = model.generation_config
generation_config.max_new_tokens = 200
generation_config.temperature = 0.7
generation_config.top_p = 0.7
generation_config.num_return_sequences = 1
generation_config.pad_token_id = tokenizer.eos_token_id
generation_config.eos_token_id = tokenizer.eos_token_id

In [ ]:
%%time
device = "cuda:0"

encoding = tokenizer(prompt, return_tensors="pt").to(device)
with torch.inference_mode():
  outputs = model.generate(
      input_ids = encoding.input_ids,
      attention_mask = encoding.attention_mask,
      generation_config = generation_config
  )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# Prep dataset

In [ ]:
from datasets import DatasetDict

In [9]:
import locale
locale.getpreferredencoding = lambda: "UTF-8"

In [30]:
data = load_dataset("bitext/Bitext-retail-banking-llm-chatbot-training-dataset")

In [32]:
data = data.remove_columns(['tags', 'category', 'intent'])

In [39]:
subset_data = data["train"].select(range(15000))  # Select the first 15,000 rows

subset_dataset = DatasetDict({"train": subset_data})

print(subset_dataset)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response'],
        num_rows: 15000
    })
})


In [40]:
def generate_prompt(data_point):
  return f"""
<human>: {data_point["instruction"]}
<assistant>: {data_point["response"]}
""".strip()

def generate_and_tokenize_prompt(data_point):
  full_prompt = generate_prompt(data_point)
  tokenized_full_prompt = tokenizer(full_prompt, padding=True, truncation=True)
  return tokenized_full_prompt

In [41]:
subset_dataset = subset_dataset["train"].shuffle().map(generate_and_tokenize_prompt)

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

# Finetune the model

In [44]:
training_args = transformers.TrainingArguments(
      per_device_train_batch_size=1,
      gradient_accumulation_steps=4,
      num_train_epochs=1,
      learning_rate=2e-4,
      fp16=True,
      save_total_limit=3,
      logging_steps=1,
      output_dir="experiments",
      optim="paged_adamw_8bit",
      lr_scheduler_type="cosine",
      warmup_ratio=0.05,
)

trainer = transformers.Trainer(
    model=model,
    train_dataset=subset_dataset,
    args=training_args,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
model.config.use_cache = False
trainer.train()

Step,Training Loss
1,1.405000
2,1.486200
3,1.495700
4,1.353700
5,1.352800
6,1.322800
7,1.558400
8,1.806300
9,1.545700
10,1.415000


TrainOutput(global_step=3750, training_loss=0.6164209331035614, metrics={'train_runtime': 9873.1307, 'train_samples_per_second': 1.519, 'train_steps_per_second': 0.38, 'total_flos': 6.693962060795443e+16, 'train_loss': 0.6164209331035614, 'epoch': 1.0})

# Save trained model

In [ ]:
model.save_pretrained("trained-model")

In [5]:
!huggingface-cli login --token ## Token should be entered here.

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `finetuning` has been saved to /home/justin.aj/.cache/huggingface/stored_tokens
Your token has been saved to /home/justin.aj/.cache/huggingface/token
Login successful.
The current active token is: `finetuning`


In [18]:
# Load adapter config
adapter_config_path = "justin-aj/bank-v1-falcon7b"
peft_config = PeftConfig.from_pretrained(adapter_config_path)

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(peft_config.base_model_name_or_path,     trust_remote_code=True,
device_map="auto")

# Load adapter
model = PeftModel.from_pretrained(base_model, "justin-aj/bank-v1-falcon7b")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path)

Loading checkpoint shards:   0%|          | 0/15 [00:00<?, ?it/s]

Some weights of FalconForCausalLM were not initialized from the model checkpoint at vilsonrodrigues/falcon-7b-instruct-sharded and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
PEFT_MODEL = "justin-aj/bank-v1-falcon7b"

model.push_to_hub(
    PEFT_MODEL, use_auth_token=True
)

In [53]:
config = PeftConfig.from_pretrained(PEFT_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    return_dict=True,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer=AutoTokenizer.from_pretrained(config.base_model_name_or_path)
tokenizer.pad_token = tokenizer.eos_token

model = PeftModel.from_pretrained(model, PEFT_MODEL)

adapter_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

/home/justin.aj/.local/lib/python3.9/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/15 [00:00<?, ?it/s]

Some weights of FalconForCausalLM were not initialized from the model checkpoint at vilsonrodrigues/falcon-7b-instruct-sharded and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


adapter_model.bin:   0%|          | 0.00/18.9M [00:00<?, ?B/s]

# Run the finetuned model

In [19]:
generation_config = model.generation_config
generation_config.max_new_tokens = 200
generation_config.temperature = 0.7
generation_config.top_p = 0.7
generation_config.num_return_sequences = 1
generation_config.pad_token_id = tokenizer.eos_token_id
generation_config.eos_token_id = tokenizer.eos_token_id

In [ ]:
# %%time
# device = "cuda:0"

# encoding = tokenizer(prompt, return_tensors="pt").to(device)
# with torch.inference_mode():
#   outputs = model.generate(
#       input_ids = encoding.input_ids,
#       attention_mask = encoding.attention_mask,
#       generation_config = generation_config
#   )

# print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [20]:
%%time
device = "cuda:0"

prompt = """
<human>: Hey"
<assistant>:
""".strip()

encoding = tokenizer(prompt, return_tensors="pt").to(device)
with torch.inference_mode():
  outputs = model.generate(
      input_ids = encoding.input_ids,
      attention_mask = encoding.attention_mask,
      generation_config = generation_config
  )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

<human>: Hey"
<assistant>: I'm here to assist you with any questions or concerns you may have about your account. To better understand your situation, could you please provide me with some additional details? This could include your account number, the specific issue you're facing, or any other relevant information that would help me provide you with the most accurate assistance. Thank you!

I'm here to help you resolve any issues you may have with your account. Feel free to share any details you think might be relevant, and I'll do my best to assist you.

If you have any other questions or need further clarification, please don't hesitate to let me know. I'm here to make sure you have a smooth experience with your account. Let's get started!

Best regards,
[Your Name]
Customer Support Specialist

Please note that I'm just a chatbot and I don't have access to your account information. However, I'm
CPU times: user 9.88 s, sys: 1.58 s, total: 11.5 s
Wall time: 11.5 s


In [21]:
%%time
device = "cuda:0"

prompt = """
<human>: How to open an account?"
<assistant>:
""".strip()

encoding = tokenizer(prompt, return_tensors="pt").to(device)
with torch.inference_mode():
  outputs = model.generate(
      input_ids = encoding.input_ids,
      attention_mask = encoding.attention_mask,
      generation_config = generation_config
  )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

<human>: How to open an account?"
<assistant>: I'm here to assist you with opening an account. Here's what you need to do:

1. Visit our website at {{Company Website URL}}.
2. Click on the "Sign Up" or "Join Now" button.
3. Fill out the registration form with your personal information, such as your name, email address, and contact details.
4. Provide any additional information requested, such as your bank account details or employment information.
5. Once you've completed the registration process, you'll receive a confirmation email or SMS.
6. Follow the instructions provided in the email or SMS to activate your account.
7. After activation, you'll be able to access your account and start using our services.

If you encounter any difficulties during the registration process or have any questions, please don't hesitate to reach out to our customer support team at {{Customer Support Phone Number}} or through the live
CPU times: user 8.99 s, sys: 1.35 s, total: 10.3 s
Wall time: 10.4 s


In [22]:
%%time
device = "cuda:0"

prompt = """
<human>: What is the use of a credit card"
<assistant>:
""".strip()

encoding = tokenizer(prompt, return_tensors="pt").to(device)
with torch.inference_mode():
  outputs = model.generate(
      input_ids = encoding.input_ids,
      attention_mask = encoding.attention_mask,
      generation_config = generation_config
  )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

<human>: What is the use of a credit card"
<assistant>: I'm here to help you with that! Using a credit card can have many benefits. Here are some of the most popular ones:

1. Convenience: Credit cards allow you to make purchases without carrying cash.
2. Building credit: Regular credit card usage can help you establish a positive credit history.
3. Rewards and perks: Many credit cards offer rewards and perks such as cashback, travel rewards, and exclusive discounts.
4. Financial flexibility: Credit cards can provide you with the flexibility to make large purchases without having to pay upfront.
5. Emergency funding: Credit cards can be used as an emergency fund in case of unexpected expenses.

Remember, it's important to use credit cards responsibly and make timely payments to avoid interest charges and maintain a good credit score. If you have any specific questions or need further assistance, feel free to let me know. I'm here to help!
CPU times: user 8.41 s, sys: 1.26 s, total: 9.6

In [23]:
%%time
device = "cuda:0"

prompt = """
<human>: How to deposit my money"
<assistant>:
""".strip()

encoding = tokenizer(prompt, return_tensors="pt").to(device)
with torch.inference_mode():
  outputs = model.generate(
      input_ids = encoding.input_ids,
      attention_mask = encoding.attention_mask,
      generation_config = generation_config
  )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

<human>: How to deposit my money"
<assistant>: I'm here to assist you with depositing your money. Here's what you need to do:

1. Log in to your account on our website or mobile app.
2. Navigate to the "Deposit" or "Add Funds" section.
3. Enter the amount you want to deposit.
4. Choose the payment method you prefer, such as a credit card, bank transfer, or online payment.
5. Follow the on-screen instructions to complete the deposit process.

If you encounter any difficulties or have any questions along the way, feel free to reach out to our customer support team at {{Customer Support Phone Number}} or through the live chat on our website at {{Company Website URL}}. We're available {{Customer Support Working Hours}} to help you out.

Please note that there may be certain restrictions or fees associated with your specific deposit method. It's always a good idea to review the terms
CPU times: user 8.65 s, sys: 1.39 s, total: 10 s
Wall time: 10.1 s


In [24]:
%%time
device = "cuda:0"

prompt = """
<human>: Working hours of the bank"
<assistant>:
""".strip()

encoding = tokenizer(prompt, return_tensors="pt").to(device)
with torch.inference_mode():
  outputs = model.generate(
      input_ids = encoding.input_ids,
      attention_mask = encoding.attention_mask,
      generation_config = generation_config
  )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

<human>: Working hours of the bank"
<assistant>: I'm here to assist you with checking the working hours of your bank. To provide you with accurate information, could you please specify which bank you are referring to? Once I have the details, I'll be able to provide you with the necessary information regarding their working hours. Thank you!
CPU times: user 2.66 s, sys: 428 ms, total: 3.09 s
Wall time: 3.1 s


In [25]:
%%time
device = "cuda:0"

prompt = """
<human>: How can I get an education loan"
<assistant>:
""".strip()

encoding = tokenizer(prompt, return_tensors="pt").to(device)
with torch.inference_mode():
  outputs = model.generate(
      input_ids = encoding.input_ids,
      attention_mask = encoding.attention_mask,
      generation_config = generation_config
  )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

<human>: How can I get an education loan"
<assistant>: I'm here to assist you with getting a education loan. Here's what you need to do:

1. Research and compare different loan options available in the market. Look for lenders who offer competitive interest rates, flexible repayment terms, and minimal requirements.

2. Determine your eligibility for a loan. This involves checking your credit score, income, and employment status. You can use online tools or consult with a financial advisor to assess your eligibility.

3. Gather all the necessary documents, such as proof of identity, income statements, and academic transcripts. This will help streamline the application process.

4. Fill out the loan application form provided by the lender. Make sure to provide accurate and complete information to avoid any delays or rejections.

5. Submit the application along with the required documents to the lender. Some lenders may require additional documentation, so be sure to follow their instruct

In [38]:
instruction = data['train'][1]['instruction']  # Input prompt
reference = data['train'][1]['response']      # Ground truth response

In [39]:
encoding = tokenizer(instruction, return_tensors="pt").to(device)
with torch.inference_mode():
    generated_ids = model.generate(
        input_ids=encoding.input_ids,
        attention_mask=encoding.attention_mask,
        max_length=512,  # Set a max length for the response
        temperature=0.7  # Adjust generation parameters as needed
    )

# Decode generated response
generated_response = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print("Generated Response:", generated_response)

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.


Generated Response: I have to activate an Visa online, how can I do it?
To activate your Visa online, you can follow these steps:

1. Visit the official website of your credit card provider.
2. Log in to your account using your username and password.
3. Navigate to the 'Account Settings' or 'Card Management' section.
4. Look for the option to activate your card.
5. Follow the prompts and provide any necessary information, such as your card details or personal identification.
6. Once you've completed the activation process, your Visa will be ready to use.

If you encounter any difficulties or have further questions, it's always a good idea to reach out to your credit card provider's customer support team. They will be able to assist you in activating your card and address any concerns you may have.


In [40]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Tokenize responses for BLEU
reference_tokens = [reference.split()]  # List of reference tokens
generated_tokens = generated_response.split()  # Generated response tokens

# Calculate BLEU score
smooth_fn = SmoothingFunction().method1
bleu_score = sentence_bleu(reference_tokens, generated_tokens, smoothing_function=smooth_fn)
print(f"BLEU Score: {bleu_score}")

BLEU Score: 0.2039157545471155


In [41]:
from rouge import Rouge

rouge = Rouge()
scores = rouge.get_scores(generated_response, reference, avg=True)
print("ROUGE Scores:", scores)

ROUGE Scores: {'rouge-1': {'r': 0.575, 'p': 0.5054945054945055, 'f': 0.538011690927123}, 'rouge-2': {'r': 0.3434343434343434, 'p': 0.272, 'f': 0.3035714236387915}, 'rouge-l': {'r': 0.55, 'p': 0.4835164835164835, 'f': 0.5146198780616258}}
